In [1]:
import torch

In [153]:
# expects x.shape = (N, d)
r2 = lambda x: torch.sum(x**2, dim=1).unsqueeze(dim=1)
model =        lambda x: torch.cos(r2(x))
model_grad =   lambda x: -2 * x * torch.sin(r2(x))
model_grad_2 = lambda x: -2 * torch.sin(r2(x)) - 4 * x**2 * torch.cos(r2(x))

In [158]:
model_grad_2(X)

tensor([[-1.1874, -1.2036, -1.8585],
        [-0.2719, -0.6062, -0.3472],
        [-1.1805, -1.2185, -2.5719],
        [-2.2092, -2.0035, -2.1265]], grad_fn=<SubBackward0>)

In [69]:
#N = 4
#D = 2
#torch.manual_seed(42)
#X = torch.rand((N,D))
def sample_domain():
    X = torch.tensor(
           [[0.9250, 0.9151, 0.32],
            [0.0815, 0.3014, 0.16],
            [0.2055, 0.2310, 0.67],
            [0.9597, 0.1719, 0.75]]
    )
    return X

X = sample_domain()
X.requires_grad_(True)

u = model(X)
print(X.shape, u.shape)
u

torch.Size([4, 3]) torch.Size([4, 1])


tensor([[-0.2228],
        [ 0.9924],
        [ 0.8554],
        [ 0.0577]], grad_fn=<UnsqueezeBackward0>)

---

In [167]:
X = sample_domain()
X.requires_grad_(True)
u = model(X)

bs, D = X.shape

# Gradient - spatial & temporatal
grad_u = torch.autograd.grad(
    inputs=X,
    outputs=u,
    grad_outputs=torch.ones_like(u),
    create_graph=True,
    retain_graph=True,
)[0]

# Laplacian - spatial only
spatial_laplace_u = torch.zeros_like(X)
hess_rows = []
for i in range(D):
    hess_row = torch.autograd.grad(
        inputs=X,
        outputs=grad_u[:,i].sum(),
        grad_outputs=torch.tensor(1.0),
        create_graph=True,
        retain_graph=True
    )[0]
    hess_rows.append(hess_row)
    spatial_laplace_u[:,i:i+1] = hess_row[:,i:i+1]

# shapes: bs x 1, bs x D, bs x 1
#return u, grad_u, spatial_laplace_u
#print(u.shape, grad_u.shape, spatial_laplace_u.shape)
#print(hess_row.shape)
print("laplace =", spatial_laplace_u)


loss = torch.mean(spatial_laplace_u**2)
#loss = torch.mean(grad_u**2)
loss.backward()
print()
print(f"loss = {loss}")
#X.grad

laplace = tensor([[-1.1874, -1.2036, -1.8585],
        [-0.2719, -0.6062, -0.3472],
        [-1.1805, -1.2185, -2.5719],
        [-2.2092, -2.0035, -2.1265]], grad_fn=<CopySlices>)

loss = 2.4820005893707275


---

In [173]:
X = sample_domain()
X.requires_grad_(True)

u, vjp_fn = torch.func.vjp(model, X) # out is just u
#print(u)
grad_u = vjp_fn(torch.ones_like(u))
#print(grad_u)

def wrapper(x):
    out, vjp_fn_w = torch.func.vjp(model, x)
    grad_out = vjp_fn_w(torch.ones_like(out))
    return grad_out[0]

out2, vjp_fn2 = torch.func.vjp(wrapper, X)
#print(out2)
vjp_fn2(torch.ones_like(out2))


(tensor([[-0.1694, -0.1885, -1.3339],
         [-0.4212, -0.8951, -0.5904],
         [-1.8140, -1.9105, -3.5726],
         [-2.4134, -2.0713, -2.3223]], grad_fn=<AddBackward0>),)

---

In [159]:
X = sample_domain()
X.requires_grad_(True)

def wrapper(x, y_fn):
    out, vjp_fn_w = torch.func.vjp(y_fn, x)
    grad_out = vjp_fn_w(torch.ones_like(out))
    return grad_out[0]

def wrapper_model(x):
    return wrapper(x, y_fn=model)

grad_u = wrapper_model(X)
#print(grad_u)

laplace_u = wrapper(X, wrapper_model)
print("laplace =", laplace_u)

loss = torch.mean(laplace_u**2)
#loss = torch.mean(grad_u**2)
loss.backward()
print()
print(f"loss = {loss}")
#X.grad

laplace = tensor([[-0.1694, -0.1885, -1.3339],
        [-0.4212, -0.8951, -0.5904],
        [-1.8140, -1.9105, -3.5726],
        [-2.4134, -2.0713, -2.3223]], grad_fn=<AddBackward0>)

loss = 3.198521614074707
